# Matching Embeddings — TF two-tower + FAISS (Food.com interactions)

Recsys-style two-tower on Food.com user<->recipe interactions
(`RAW_interactions.csv`, 1.13M rows, 2.2e-5 density). Positive = rating >= 4
implicit feedback, negatives sampled uniformly from recipes. Serves
`app/embedding_engine.py` `/api/v1/embeddings/match`. Artifacts: the
`matching_embeddings` ONNX user-tower and the `matching_items.faiss` index plus
ID maps for serving-side lookup.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'faiss'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


In [ ]:
# Load interactions; implicit feedback = rating >= 4
import os
import numpy as np
import pandas as pd
from buddy_data import food_com_interactions

N = {'smoke': 60_000, 'demo': 300_000, 'full': 1_000_000}[SCALE]
it = food_com_interactions()
it = it[it['rating'] >= 4]
rng = np.random.default_rng(42)
it = it.sample(n=min(N, len(it)), random_state=42)
print('sampled interactions:', it.shape)

u_cat = it['user_id'].astype('category')
i_cat = it['recipe_id'].astype('category')
it['u'] = u_cat.cat.codes.values
it['i'] = i_cat.cat.codes.values
n_users, n_items = u_cat.cat.categories.size, i_cat.cat.categories.size
print('users:', n_users, 'items:', n_items)

In [ ]:
# Build pos/neg pairs (1 positive : 1 uniform-random negative)
users = np.concatenate([it['u'], it['u']]).astype(np.int64)
items = np.concatenate([it['i'], rng.integers(0, n_items, size=len(it))]).astype(np.int64)
labels = np.concatenate([np.ones(len(it)), np.zeros(len(it))]).astype(np.float32)
idx = rng.permutation(len(users))
users, items, labels = users[idx], items[idx], labels[idx]

split = int(0.9 * len(users))
(Xu_tr, Xi_tr, Y_tr) = users[:split], items[:split], labels[:split]
(Xu_va, Xi_va, Y_va) = users[split:], items[split:], labels[split:]
print('train pairs:', len(Y_tr), '| val pairs:', len(Y_va))

In [ ]:
# Two-tower with logit output (recompile for BCE)
import tensorflow as tf
from tf_utils import build_two_tower

m = build_two_tower(embed_dim=64, n_users=n_users, n_items=n_items)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
m.summary()

In [ ]:
# Train (embedding lookups — fast on CPU)
import time
EPOCHS = {'smoke': 1, 'demo': 5, 'full': 10}[SCALE]
t0 = time.time()
m.fit([Xu_tr, Xi_tr], Y_tr, epochs=EPOCHS, batch_size=1024,
      validation_data=([Xu_va, Xi_va], Y_va), verbose=1)
print(f'train {time.time()-t0:.0f}s')

In [ ]:
# Ranking eval: HR@10 / MRR over 100 random distractors for held-out users
import numpy as np

user_emb = m.layers[2].get_weights()[0]      # (n_users, 64)
item_emb = m.layers[3].get_weights()[0]      # (n_items, 64)
pos_pairs = np.unique(np.stack([Xu_va, Xi_va], axis=1), axis=0)

rng = np.random.default_rng(0)
hits, rr, n_eval = 0, 0.0, 0
for u, i in pos_pairs[:2000]:
    cand = np.concatenate([[i], rng.integers(0, n_items, size=99)])
    scores = item_emb[cand] @ user_emb[u]
    order = np.argsort(scores)[::-1]               # rank 0 = best
    rank = int(np.where(order == 0)[0][0]) + 1
    hits += int(rank <= 10)
    rr += 1.0 / rank
    n_eval += 1
print(f'HR@10={hits/n_eval:.3f}  MRR={rr/n_eval:.3f}  (n={n_eval})')

### Export contract (consumed by the AI service)

The cells below write `../models/matching_embeddings.onnx` and its dynamic-INT8 quantized copy
`matching_embeddings_int8.onnx`. `app/ml/serving.py::load_preferred('matching_embeddings')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [ ]:
# Build + persist FAISS item index and ID maps for serving
import faiss

item_emb = item_emb.astype('float32')
faiss.normalize_L2(item_emb)
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb)
faiss.write_index(index, '../models/matching_items.faiss')
np.save('../models/matching_users.npy', u_cat.cat.categories.to_numpy())
np.save('../models/matching_recipes.npy', i_cat.cat.categories.to_numpy())
print('FAISS index items:', index.ntotal)

# quick sanity: top-5 for a random user
u = int(pos_pairs[0][0])
print('top5 for user', u_cat.cat.categories[u], '->',
      [int(x) for x in index.search(user_emb[u:u+1].astype('float32'), 5)[1][0]])

In [ ]:
# Export the user tower as ONNX (+ INT8) for online matching
from pathlib import Path
import tensorflow as tf
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

emb_dim = m.layers[2].output.shape[-1]
tinp = tf.keras.Input(shape=(1,), dtype='int64')
tx = tf.keras.layers.Embedding(n_users, emb_dim,
                               weights=[m.layers[2].get_weights()[0]])(tinp)
tower = tf.keras.Model(tinp, tf.keras.layers.Flatten()(tx))
onnx = export_keras_onnx(tower, Path('../models'), 'matching_embeddings', '1.0.0',
                         input_signature=[tf.TensorSpec((None, 1), tf.int64, name='user_id')])
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'matching_embeddings', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'hr@10': round(hits / n_eval, 4), 'mrr': round(rr / n_eval, 4)}})
print('exported', q)